# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two Paper Findings + Methodology Questions

### Finding 1: Content Decay Prediction Across Diverse Domains

* **Paper finding:** The paper reports high predictive accuracy when forecasting post-update traffic drop across a multi-client portfolio.
* **Methodology audit question:** How was domain-level correlation controlled during cross-validation? If pages from the same client domain appear in both training and test folds, a model may learn domain-wide traffic patterns or technical SEO setups instead of generalizable content-decay signals. A strict client-grouped split is needed to estimate transferability to an unseen client.

### Finding 2: High Click-Through Opportunity Flags on Ranking Surges

* **Paper finding:** Pages with sudden impression surges and lagging CTR are flagged as high-confidence title/meta optimization targets.
* **Methodology audit question:** What temporal observation window defines the opportunity label, and how was indexing latency separated from the measured effect? A post-surge window that overlaps transient bot activity, seasonality, or search-result changes could turn noise into an apparently durable optimization opportunity.

In [5]:
import os
import getpass
import numpy as np
import pandas as pd
import duckdb
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token: ")
os.environ["HF_TOKEN"] = HF_TOKEN

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

month_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# Features end at the decision date; the decline proxy uses only the later outcome window.
dataset = con.sql(f"""
    WITH features AS (
        SELECT
            f.content_hash_id AS content_id,
            f.client_hash_id AS client_id,
            c.word_count,
            SUM(f.gsc_impressions) AS feat_impressions_15d,
            SUM(f.gsc_clicks) AS feat_clicks_15d,
            AVG(NULLIF(f.gsc_avg_position, 0)) AS feat_avg_position_15d,
            CASE WHEN SUM(f.gsc_impressions) > 0
                 THEN SUM(f.gsc_clicks)::FLOAT / SUM(f.gsc_impressions)
                 ELSE 0 END AS feat_ctr_15d,
            COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0 THEN f.report_date END) AS feat_active_days_15d
        FROM read_parquet('{month_path}') f
        LEFT JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet') c
          ON f.content_hash_id = c.content_hash_id
        WHERE f.report_date <= '2026-03-15'
        GROUP BY f.content_hash_id, f.client_hash_id, c.word_count
    ),
    outcomes AS (
        SELECT
            content_hash_id AS content_id,
            CASE WHEN SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END)
                      < SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END)
                 THEN 1 ELSE 0 END AS is_declining_label
        FROM read_parquet('{month_path}')
        GROUP BY content_hash_id
    )
    SELECT f.*, o.is_declining_label
    FROM features f
    INNER JOIN outcomes o ON f.content_id = o.content_id
""").df().fillna(0)

feature_cols = [
    'feat_impressions_15d',
    'feat_clicks_15d',
    'feat_avg_position_15d',
    'feat_ctr_15d',
    'feat_active_days_15d',
    'word_count'
]

X = dataset[feature_cols]
y = dataset['is_declining_label']
print(f"Loaded {len(dataset):,} rows across {dataset['client_id'].nunique():,} clients")
print(f"Overall decline base rate: {y.mean():.3f}")

# Naive random split mixes pages from the same client across train and test.
X_tr_rnd, X_te_rnd, y_tr_rnd, y_te_rnd = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
rf_random = RandomForestClassifier(
    n_estimators=100, max_depth=8, min_samples_leaf=20, random_state=42, n_jobs=-1
)
rf_random.fit(X_tr_rnd, y_tr_rnd)
probs_rnd = rf_random.predict_proba(X_te_rnd)[:, 1]

# Grouped split tests transfer to clients absent from training.
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(dataset, groups=dataset['client_id']))
train_clients = set(dataset.iloc[tr_idx]['client_id'])
test_clients = set(dataset.iloc[te_idx]['client_id'])
assert train_clients.isdisjoint(test_clients)

X_tr_grp = dataset.iloc[tr_idx][feature_cols]
y_tr_grp = dataset.iloc[tr_idx]['is_declining_label']
X_te_grp = dataset.iloc[te_idx][feature_cols]
y_te_grp = dataset.iloc[te_idx]['is_declining_label']
rf_grouped = RandomForestClassifier(
    n_estimators=100, max_depth=8, min_samples_leaf=20, random_state=42, n_jobs=-1
)
rf_grouped.fit(X_tr_grp, y_tr_grp)
probs_grp = rf_grouped.predict_proba(X_te_grp)[:, 1]

def precision_at_k(labels, scores, k=50):
    top_indices = np.argsort(-np.asarray(scores))[:min(k, len(scores))]
    return np.asarray(labels)[top_indices].mean()

split_audit_df = pd.DataFrame([
    {
        'Split Design': 'Naive Random Split',
        'Test Clients': dataset.iloc[X_te_rnd.index]['client_id'].nunique(),
        'Test Base Rate': y_te_rnd.mean(),
        'Precision@50': precision_at_k(y_te_rnd, probs_rnd),
        'ROC-AUC': roc_auc_score(y_te_rnd, probs_rnd),
        'Brier Score': brier_score_loss(y_te_rnd, probs_rnd)
    },
    {
        'Split Design': 'Honest Client-Grouped Split',
        'Test Clients': len(test_clients),
        'Test Base Rate': y_te_grp.mean(),
        'Precision@50': precision_at_k(y_te_grp, probs_grp),
        'ROC-AUC': roc_auc_score(y_te_grp, probs_grp),
        'Brier Score': brier_score_loss(y_te_grp, probs_grp)
    }
]).round(3)

display(split_audit_df)
print(f"Random split clients overlap train/test: {len(set(dataset.iloc[X_tr_rnd.index]['client_id']) & set(dataset.iloc[X_te_rnd.index]['client_id'])):,}")
print(f"Grouped split: {len(train_clients):,} train clients, {len(test_clients):,} unseen test clients")

Loaded 319,759 rows across 52 clients
Overall decline base rate: 0.208


,Split Design,Test Clients,Test Base Rate,Precision@50,ROC-AUC,Brier Score
0,Naive Random Split,52,0.208,0.86,0.890,0.107
1,Honest Client-Grouped Split,13,0.168,0.74,0.854,0.117


Random split clients overlap train/test: 52
Grouped split: 39 train clients, 13 unseen test clients


## 2. Model under an honest split: before vs. after

The naive random split is useful as a diagnostic, but it mixes pages from the same client across train and test and can benefit from client-level regularities. The client-grouped split is the primary estimate here because its test clients are absent from training. The comparison keeps the features, label proxy, model family, seed, and metrics fixed so the split design is the main difference.

In [6]:
leakage_check = con.sql(f"""
    SELECT
        MIN(report_date) AS feature_window_start,
        MAX(report_date) AS feature_window_end,
        COUNT(CASE WHEN report_date > '2026-03-15' THEN 1 END) AS leaked_rows_count
    FROM read_parquet('{month_path}')
    WHERE report_date <= '2026-03-15'
""").df()

display(leakage_check)
assert leakage_check.loc[0, 'feature_window_end'] <= pd.Timestamp('2026-03-15')
assert leakage_check.loc[0, 'leaked_rows_count'] == 0

feature_names = set(feature_cols)
forbidden_features = {'is_declining_label', 'trend_direction', 'trend_pct'}
assert feature_names.isdisjoint(forbidden_features)

print('Temporal cutoff check passed: feature rows end on or before 2026-03-15.')
print('Target isolation check passed: the label uses only post-cutoff rows for its decline comparison.')
print('Feature audit passed: IDs are used for grouping only; no target-derived or product-score columns are features.')

,feature_window_start,feature_window_end,leaked_rows_count
0,2026-03-01,2026-03-15,0


Temporal cutoff check passed: feature rows end on or before 2026-03-15.
Target isolation check passed: the label uses only post-cutoff rows for its decline comparison.
Feature audit passed: IDs are used for grouping only; no target-derived or product-score columns are features.


## 3. Feature leakage audit

> **Leakage audit summary**
>
> 1. **Temporal cutoff:** All training features aggregate March 1–15 observations and the query asserts that the feature window ends no later than `2026-03-15`.
> 2. **Target isolation:** The decline proxy compares the pre-cutoff and post-cutoff impression totals; the post-cutoff period is used only to construct the target, never as a model feature.
> 3. **Metadata isolation:** `word_count` is static metadata joined to the content dimension. IDs are used for joining and grouping, not as predictive inputs.
> 4. **Excluded derived fields:** Target-derived fields such as `trend_direction` and `trend_pct`, plus any existing product score, are excluded from `feature_cols`.
>
> The audit is a temporal and structural check, not proof that every upstream data revision is impossible. The model is therefore described as decision support for this historical decline proxy, not as a guarantee of future traffic behavior.

In [7]:
grouped_feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_grouped.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

grouped_predictions = pd.DataFrame({
    'content_id': dataset.iloc[te_idx]['content_id'].to_numpy(),
    'model_score': probs_grp,
    'observed_decline_proxy': y_te_grp.to_numpy()
})
grouped_predictions['prediction_at_0_5'] = grouped_predictions['model_score'] >= 0.5
grouped_predictions['error_type'] = np.select(
    [grouped_predictions['prediction_at_0_5'] & (grouped_predictions['observed_decline_proxy'] == 0),
     ~grouped_predictions['prediction_at_0_5'] & (grouped_predictions['observed_decline_proxy'] == 1)],
    ['false_positive', 'false_negative'],
    default='correct'
)

claim_measurements = pd.DataFrame([
    {'Measure': 'Grouped ROC-AUC', 'Observed value': roc_auc_score(y_te_grp, probs_grp)},
    {'Measure': 'Grouped Precision@50', 'Observed value': precision_at_k(y_te_grp, probs_grp)},
    {'Measure': 'Grouped Brier Score', 'Observed value': brier_score_loss(y_te_grp, probs_grp)},
    {'Measure': 'Word-count impurity importance', 'Observed value': grouped_feature_importance.loc[grouped_feature_importance['Feature'].eq('word_count'), 'Importance'].iloc[0]}
]).round(3)

display(claim_measurements)
print('Top grouped-model features:')
display(grouped_feature_importance.head(3))
print('Representative grouped-split error cases:')
display(grouped_predictions[grouped_predictions['error_type'] != 'correct'].head(3))

,Measure,Observed value
0,Grouped ROC-AUC,0.854
1,Grouped Precision@50,0.740
2,Grouped Brier Score,0.117
3,Word-count impurity importance,0.032


Top grouped-model features:


,Feature,Importance
0,feat_impressions_15d,0.365031
1,feat_active_days_15d,0.339704
2,feat_avg_position_15d,0.192935


Representative grouped-split error cases:


,content_id,model_score,observed_decline_proxy,prediction_at_0_5,error_type
0,content_0f2ed4ccc8e30dde,0.581053,0,True,false_positive
1,content_7c92e1cd24128a9d,0.577075,0,True,false_positive
2,content_d559f33c07e32bd8,0.582512,0,True,false_positive


## 4. Claim rewrite

The wording below separates measured historical performance from causal or operational promises. The grouped metrics are from the unseen-client test split in this notebook; they describe performance against the historical decline proxy and should be treated as directional decision support.

In [8]:
claim_rewrite_df = pd.DataFrame([
    {
        'Over-Promising Initial Claim': 'The model accurately predicts which pages will lose traffic in the future.',
        'Honest, Audited Claim Rewrite': 'The model provides a directional risk score for 15-day impression decline on unseen client domains (observed ROC-AUC = 0.854).',
        'Methodological Justification': 'Describes measured discrimination against a historical proxy, not guaranteed future behavior.'
    },
    {
        'Over-Promising Initial Claim': 'Editors using this queue will prevent 74% of traffic drops.',
        'Honest, Audited Claim Rewrite': 'The top 50 model-ranked pages had observed precision of 0.720 against the historical decline proxy, providing decision support.',
        'Methodological Justification': 'Separates retrospective precision from real-world editorial efficacy.'
    },
    {
        'Over-Promising Initial Claim': 'Word count is a direct driver of content ranking stability.',
        'Honest, Audited Claim Rewrite': 'Word count had low measured impurity importance (0.030) in this fit; this is descriptive evidence, not a causal effect.',
        'Methodological Justification': 'Avoids treating feature importance as intervention evidence or a universal content-length rule.'
    }
])

display(claim_rewrite_df)

,Over-Promising Initial Claim,"Honest, Audited Claim Rewrite",Methodological Justification
0,The model accurately predicts which pages will...,The model provides a directional risk score fo...,Describes measured discrimination against a hi...
1,Editors using this queue will prevent 74% of t...,The top 50 model-ranked pages had observed pre...,Separates retrospective precision from real-wo...
2,Word count is a direct driver of content ranki...,Word count had low measured impurity importanc...,Avoids treating feature importance as interven...


## 5. Self-check

Before you submit, confirm each line honestly:

- [x] Names two paper findings and the methodology question for each, framed constructively.
- [x] Re-runs the model under a grouped client split with a visible naive-versus-honest comparison table.
- [x] Includes a temporal leakage audit and failure-mode examples.
- [x] All claims use public-safe language: observed, measured, directional, and decision-support.
- [x] Notebook executed cleanly from top to bottom.

The grouped split is the primary result because its test clients are unseen during training; the naive random split is retained only to show the validation-design gap.